# Vorbereitungen

Bevor die Skripte 01 bis 09 laufen, müssen die verwendeten Softwarepakete, die Sprachmodelle von spaCy und die Daten installiert sein. Dieses Notebook muss nur einmal pro Python-Umgebung ausgeführt werden; Schritt 5 (API-Schlüssel) lässt sich auch später nachholen.

| Paket | Verwendung |
|---|---|
| `pandas`, `numpy`, `pyarrow` | Tabellen und Arrays; Parquet-Dateien für die Daten |
| `matplotlib` | Grafiken |
| `nltk` | Tokenisierung und Stoppwortliste (Skript 01) |
| `spacy` | linguistische Annotation, Wortvektoren und Satzsegmentierung (Skripte 02-04, 06, 08, 09) |
| `scikit-learn` | Dokument-Term-Matrix, TF-IDF, Clustering, Topic Models, Klassifikation (Skripte 03-09) |
| `gensim` | eigene Wortvektoren mit word2vec und fastText (Skript 06) |
| `torch`, `sentence-transformers` | Sentence Transformer und kontextuelle Embeddings (Skripte 06, 07, 09) |
| `umap-learn` | Projektion der Embeddings in zwei Dimensionen (Skript 06) |
| `anthropic` | API von Anthropic für Claude (Skripte 07 und 09) |
| `mistralai` | API von Mistral (Skript 09) |
| `jupyterlab` | Notebooks ausführen |

## 1. Pakete installieren

Der Befehl `%pip install` installiert die Pakete in genau die Python-Umgebung, in der dieser Jupyter-Kernel läuft. Alternativ lässt sich derselbe Befehl ohne `%` in einem Terminal ausführen: `python -m pip install ...`.

In [ ]:
%pip install anthropic gensim jupyterlab matplotlib mistralai nltk numpy pandas pyarrow scikit-learn sentence-transformers spacy torch umap-learn

Die Installation dauert beim ersten Mal einige Minuten, vor allem wegen `torch`. Sind die Pakete schon vorhanden, meldet pip für jedes Paket `Requirement already satisfied`. Damit neu installierte Pakete gefunden werden, sollte der Kernel danach neu gestartet werden (*Kernel → Restart Kernel*).

Unter Windows und macOS installiert pip `torch` in der Variante für die CPU, die auf jedem Rechner läuft (unter Linux kommt die Variante für NVIDIA-Grafikkarten). Wer unter Windows eine NVIDIA-Grafikkarte nutzen möchte, installiert `torch` nach der Anleitung auf https://pytorch.org/get-started/locally/; die Skripte verwenden die Grafikkarte dann automatisch. Für dieses Korpus ist eine Grafikkarte nicht nötig.

## 2. Sprachmodelle von spaCy

spaCy braucht für jede Sprache ein trainiertes Modell. Die Skripte verwenden drei deutsche Modelle in verschiedenen Größen: `sm` (klein, Skript 02), `md` (mittel, die meisten Skripte) und `lg` (groß, wegen seiner Wortvektoren in Skript 06). Zusammen sind sie etwa 700 MB groß.

Die Zelle lädt nur Modelle herunter, die noch fehlen, und verwendet dafür genau das Python des Kernels (`sys.executable`). Ein `!python …` in einer Zelle könnte dagegen ein anderes Python auf dem Rechner erwischen.

In [ ]:
import importlib.util
import subprocess
import sys

for model in ["de_core_news_sm", "de_core_news_md", "de_core_news_lg"]:
    if importlib.util.find_spec(model) is None:
        print(f"lade {model} ...")
        subprocess.run([sys.executable, "-m", "spacy", "download", model], check=True)
    else:
        print(f"{model} ist schon installiert")

## 3. Installation prüfen

Die folgende Zelle lädt alle Pakete und Modelle und gibt ihre Versionen aus. Erscheint ein `ModuleNotFoundError` oder `OSError`, ist das betreffende Paket oder Modell nicht installiert - meist hilft ein Neustart des Kernels nach Schritt 1 oder 2.

In [ ]:
from importlib.metadata import version

import anthropic, gensim, matplotlib, mistralai, nltk, numpy, pandas, pyarrow, sentence_transformers, sklearn, spacy, torch, umap

for paket in ["anthropic", "gensim", "matplotlib", "mistralai", "nltk", "numpy", "pandas", "pyarrow",
              "scikit-learn", "sentence-transformers", "spacy", "torch", "umap-learn"]:
    print(f"{paket:22} {version(paket)}")

for modell in ["de_core_news_sm", "de_core_news_md", "de_core_news_lg"]:
    print(f"{modell:22} {spacy.load(modell).meta['version']}")
print("Grafikkarte (CUDA):    ", "ja" if torch.cuda.is_available() else "nein, die Skripte laufen auf der CPU")

## 4. Daten herunterladen

Die Daten liegen nicht im Repository. Die folgende Zelle lädt sie herunter (rund 1 GB: das One Million Posts Corpus und den deutschen Teil von MLSUM), behält nur, was der Kurs braucht, und speichert es als drei Parquet-Dateien (zusammen knapp 100 MB) im Ordner `data/`. Das dauert je nach Verbindung einige Minuten. Sind die Dateien schon da, passiert nichts. Mehr zu den Daten und ihren Lizenzen steht in der README und in Skript 01.

In [ ]:
import textmining as tm

tm.prepare_data()
corpus, posts = tm.load_corpus(), tm.load_posts()
print(f"{len(corpus):,} Schlagzeilen, {len(posts):,} annotierte Leserkommentare")
print(corpus["outlet"].value_counts().to_string())

## 5. API-Schlüssel (nur für die Skripte 07 und 09)

Die Skripte 07 und 09 vergleichen die übrigen Verfahren mit großen Sprachmodellen: Skript 07 nutzt Claude (Anthropic), Skript 09 Claude und Mistral. Dafür braucht es API-Schlüssel. Trage sie in die Datei **`api-keys.env`** im Ordner des Kurses ein:

```
ANTHROPIC_API_KEY=sk-ant-...
MISTRAL_API_KEY=...
```

Fehlt die Datei, kopiere die Vorlage `api-keys.example.env` nach `api-keys.env`. Die Datei ist in `.gitignore` eingetragen und wird nie committet; gib sie nicht weiter. Ohne Schlüssel laufen beide Skripte trotzdem durch, nur ohne die LLM-Ergebnisse. Wird ein Schlüssel in der Datei ausgetauscht, genügt es, die Zelle mit `tm.load_api_keys()` erneut auszuführen. Die folgende Zelle zeigt, welche Schlüssel gefunden wurden - die Schlüssel selbst gibt sie nicht aus.

In [ ]:
tm.load_api_keys()